# GPU Setup & Verification Notebook
**Drug Discovery Platform - RTX 4050 (6GB VRAM)**

This notebook helps you:
1. Verify GPU detection
2. Test PyTorch CUDA support  
3. Run sample ML workloads on GPU
4. Monitor performance improvements

**Prerequisites:**
- NVIDIA Driver 591.44+ (you have ✅)
- PyTorch with CUDA installed

In [ ]:
# Check NVIDIA driver and GPU
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

## Step 1: Verify PyTorch Installation

Run the cell below to check if PyTorch is installed and CUDA-enabled.

In [ ]:
try:
    import torch
    print(f"✅ PyTorch Version: {torch.__version__}")
    print(f"✅ CUDA Available: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"✅ CUDA Version: {torch.version.cuda}")
        print(f"✅ GPU Name: {torch.cuda.get_device_name(0)}")
        print(f"✅ VRAM Total: {round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)} GB")
    else:
        print("❌ GPU NOT DETECTED")
        print("\n📝 To fix, run in terminal:")
        print("pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126")
        
except ImportError:
    print("❌ PyTorch not installed")
    print("\n📝 To install, run in terminal:")
    print("pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126")

## Step 2: Test Device Manager

Check if the custom DeviceManager detects your GPU.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

from apps.neural_networks.device_manager import DeviceManager, get_device, device_info

# Print device status
DeviceManager.print_status()

# Get device info as dict
info = device_info()
print("\n📊 Device Info Dictionary:")
for key, value in info.items():
    print(f"  {key}: {value}")

## Step 3: GPU Memory Test

Create tensors on GPU and monitor memory usage.

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    
    print("Creating test tensors on GPU...")
    
    # Small tensor
    x = torch.randn(1000, 1000, device=device)
    print(f"✅ Tensor shape: {x.shape}")
    print(f"   Memory allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    
    # Larger tensor
    y = torch.randn(5000, 5000, device=device)
    print(f"\n✅ Larger tensor shape: {y.shape}")
    print(f"   Memory allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    print(f"   Memory cached: {torch.cuda.memory_reserved(0) / 1024**2:.2f} MB")
    
    # Matrix multiplication (GPU accelerated)
    import time
    start = time.time()
    z = torch.mm(x, x.T)
    torch.cuda.synchronize()  # Wait for GPU to finish
    elapsed = time.time() - start
    
    print(f"\n🚀 Matrix multiplication (1000x1000): {elapsed*1000:.2f} ms")
    
    # Cleanup
    del x, y, z
    torch.cuda.empty_cache()
    print("\n🗑️ GPU cache cleared")
    
else:
    print("⏭️ Skipping - GPU not available")

## Step 4: Compare CPU vs GPU Performance

Run a simple neural network on both CPU and GPU to see speedup.

In [ ]:
import torch
import torch.nn as nn
import time

# Define a simple model
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1000, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Test on CPU
print("🔄 Testing on CPU...")
model_cpu = SimpleNet()
x_cpu = torch.randn(128, 1000)

start = time.time()
for _ in range(100):
    output = model_cpu(x_cpu)
cpu_time = time.time() - start
print(f"   CPU Time: {cpu_time:.3f} seconds")

# Test on GPU (if available)
if torch.cuda.is_available():
    print("\n🚀 Testing on GPU...")
    model_gpu = SimpleNet().cuda()
    x_gpu = torch.randn(128, 1000, device='cuda')
    
    # Warm up
    for _ in range(10):
        output = model_gpu(x_gpu)
    torch.cuda.synchronize()
    
    start = time.time()
    for _ in range(100):
        output = model_gpu(x_gpu)
    torch.cuda.synchronize()
    gpu_time = time.time() - start
    print(f"   GPU Time: {gpu_time:.3f} seconds")
    
    speedup = cpu_time / gpu_time
    print(f"\n⚡ Speedup: {speedup:.2f}x faster on GPU!")
    
    # Cleanup
    del model_gpu, x_gpu
    torch.cuda.empty_cache()
else:
    print("\n⏭️ Skipping GPU test - not available")

## Step 5: Test Drug Discovery Models

Test the actual SMILES generator with GPU acceleration.

In [ ]:
# Test with actual project code
import sys
sys.path.insert(0, os.path.dirname(os.getcwd()))

try:
    from ml_smiles_generator_pytorch import SmilesGenerator
    
    print("🧪 Testing SMILES Generator with GPU...")
    generator = SmilesGenerator(max_length=50, embedding_dim=64, hidden_dim=128)
    
    # Show device info
    print(f"\n✅ Generator initialized on: {generator.device}")
    
    # Test with sample data
    sample_smiles = [
        "CCO",  # Ethanol
        "CC(C)Cc1ccc(cc1)C(C)C(O)=O",  # Ibuprofen
        "CC(=O)Oc1ccccc1C(=O)O"  # Aspirin
    ]
    
    print(f"\n📊 Sample SMILES for testing:")
    for i, smi in enumerate(sample_smiles, 1):
        print(f"  {i}. {smi}")
    
    print("\n✅ SMILES Generator ready for GPU-accelerated training!")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nMake sure to install PyTorch with CUDA first")

## Step 6: Monitor GPU Usage

Use this cell to check real-time GPU stats while training models.

In [ ]:
import torch

if torch.cuda.is_available():
    print("🎯 GPU Status:")
    print(f"  Device: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"  VRAM Used: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"  VRAM Cached: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    print(f"  VRAM Free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1024**3:.2f} GB")
    
    # Utilization (requires pynvml)
    try:
        import pynvml
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        util = pynvml.nvmlDeviceGetUtilizationRates(handle)
        temp = pynvml.nvmlDeviceGetTemperature(handle, 0)
        
        print(f"\n  GPU Utilization: {util.gpu}%")
        print(f"  Memory Utilization: {util.memory}%")
        print(f"  Temperature: {temp}°C")
        pynvml.nvmlShutdown()
    except ImportError:
        print("\n  💡 Install pynvml for detailed stats: pip install pynvml")
else:
    print("⏭️ GPU not available")

## Summary

✅ **If all cells run successfully, your GPU is ready for:**
- ML model training (PyTorch)
- SMILES generation
- Drug property prediction
- Neural network inference

### Performance Tips for 6GB VRAM:
1. Use `batch_size=4` for medium models
2. Enable mixed precision: `torch.cuda.amp`
3. Clear cache between runs: `torch.cuda.empty_cache()`
4. Use 4-bit quantization for large models

### Monitoring GPU:
In terminal, run: `nvidia-smi -l 1` (updates every second)

---

**Next:** Run Django server and test ML endpoints with GPU acceleration!